# PatchTST — full-month training run (Google Colab)

Trains the **PatchTST** baseline for [BipowerQuant](https://github.com/NynsenFaber/BipowerQuant)
on the complete May 2026 BTC/USDT tape, benchmarks it against the Logistic Regression
and XGBoost baselines, and exports a checkpoint you can score locally.

### Before you press Run all

1. `Runtime → Change runtime type → Hardware accelerator: **T4 GPU**`
2. Push the branch holding this code to the remote, so cell 1 can clone it.

Then **Runtime → Run all**. It pauses exactly once, at the very start, for Google
Drive authorisation — click through it and the rest is unattended. (Set
`USE_DRIVE = False` in cell 2 to skip that, at the cost of losing the bar cache
and weights if the runtime disconnects.)

### Rough budget

| Stage | Time |
| :--- | :--- |
| Download + unzip the month | ~2 min |
| Fold 72M ticks into 2.7M second-bars | ~1 min (cached afterwards) |
| Training, `TRAIN_STRIDE = 6` | ~3 min/epoch, early-stops well before the 30-epoch cap |
| Inference benchmark | ~2 min |

Cell 6 measures real training throughput and prints an epoch estimate *before* the
training cell runs, so you can stop and lower the budget if it looks wrong.

### What is being tested

The same classification problem the tabular baselines solve — given a 5-minute
lookback, will the next minute return more than the 5 bp fee threshold? — with one
difference: PatchTST reads the 300 raw one-second bars as 6 parallel channels
instead of 7 hand-aggregated scalars. Section 9 scores all three models on
*identical* windows so the comparison is exact.

All modelling logic lives in the repository (`python/sequence_matrix.py`,
`python/patchtst_model.py`, `python/patchtst_train.py`, `python/benchmark_inference.py`);
this notebook only sets knobs and calls them.

## 0. Confirm the GPU

In [ ]:
!nvidia-smi -L || echo "No GPU attached — Runtime > Change runtime type > T4 GPU"

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB",
          "| bf16:", torch.cuda.is_bf16_supported())
    # Free speed on Ampere and newer; a no-op on T4.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

## 1. Pull the code

The branch below has to exist **on the remote** — push it before running this.

*No remote access?* Skip this cell, upload `sequence_matrix.py`, `patchtst_model.py`,
`patchtst_train.py` and `data_feeder.py` through the Colab file browser into
`/content/`, and run `sys.path.insert(0, "/content")` instead.

In [ ]:
REPO_URL = "https://github.com/NynsenFaber/BipowerQuant.git"
BRANCH   = "add-patchTST"
REPO_DIR = "/content/BipowerQuant"

import os, subprocess, sys

%pip install -q polars

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

sys.path.insert(0, f"{REPO_DIR}/python")

import sequence_matrix as seq
from patchtst_model import (
    PatchTSTConfig, PatchTSTClassifier, WindowBatcher,
    binary_metrics, format_metrics, predict_proba, save_checkpoint, threshold_sweep,
)
from patchtst_train import TrainConfig, estimate_throughput, fit, pick_device

print("loaded", seq.__file__)

## 2. Fetch the tape

Pulled straight from the Binance public archive, so nothing has to be uploaded.
`BTCUSDT-trades-2026-05.zip` is ~1.5 GB compressed / ~5 GB unpacked and takes a
couple of minutes.

Mounting Drive is optional but recommended: the 1-second bar cache and the trained
weights get written there, so a disconnected runtime does not cost you the download
or the training run.

In [ ]:
SYMBOL, MONTH = "BTCUSDT", "2026-05"
DATA_DIR      = "/content/data"
USE_DRIVE     = True                                  # cache bars + weights on Drive
DRIVE_DIR     = "/content/drive/MyDrive/BipowerQuant"

import os

os.makedirs(DATA_DIR, exist_ok=True)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

CSV_PATH = f"{DATA_DIR}/{SYMBOL}-trades-{MONTH}.csv"
if not os.path.exists(CSV_PATH):
    url = f"https://data.binance.vision/data/spot/monthly/trades/{SYMBOL}/{SYMBOL}-trades-{MONTH}.zip"
    !wget -q --show-progress -O /content/trades.zip {url}
    !unzip -o -q /content/trades.zip -d {DATA_DIR}
    !rm -f /content/trades.zip

print(CSV_PATH, f"{os.path.getsize(CSV_PATH) / 1e9:.2f} GB")

## 3. Configuration

`LABEL_MODE = "triple_barrier"` is the target the project actually cares about:
from the moment a window closes, does the price touch `+BARRIER` or `-BARRIER`
first, within `HORIZON` seconds? Both classes require the same 5 bp move, so a
volatility forecast scores 0.5 on it and only directional skill can beat that.
Windows where neither barrier is touched inside the horizon (~79% of them, on
BTC/USDT) carry no side and are dropped from the population — which is why
`TRAIN_STRIDE` is 2 here rather than the 6 the old dense target needed.

`LABEL_MODE = "fee_threshold"` restores the original `forward_return > +5bp`
target if you want to reproduce the pre-audit numbers.

`HOURS = None` takes the whole month: ~2.68M bars and a test split of ~6 days.
An 8-hour run leaves ~95 minutes of test data, and since each observation spans
360 s that is only ~16 independent episodes — enough to check the pipeline runs,
nowhere near enough to separate an edge from noise. Use 8 hours to debug and the
month to conclude.


In [ ]:
# ---- problem definition (keep matched to python/sequence_matrix.py) ----
WINDOW      = 300              # 5-minute lookback, in 1-second bars = PatchTST's sequence length L
HORIZON     = 60               # vertical barrier: decide within 60 seconds
BARRIER     = 0.0005           # +/-5 bp horizontal barriers, one taker round-trip
LABEL_MODE  = "triple_barrier" # "fee_threshold" reproduces the pre-audit target
CHANNEL_SET = "raw"            # "raw" = log_return + ofi; "full" = the old 6 channels

# ---- how much tape ----
HOURS          = None       # None = the whole month; 8.0 = the fast sanity run
SKIP_HOURS     = 0.0

# ---- splits (chronological, purged at the boundaries) ----
TRAIN_FRAC, VAL_FRAC = 0.70, 0.10   # test is the remaining 20%
TRAIN_STRIDE         = 2            # the barrier filter already drops ~79% of windows
VAL_STRIDE           = 2

# ---- model ----
PATCH_LEN, PATCH_STRIDE = 16, 8     # P and S from the paper -> 37 tokens instead of 300 bars
D_MODEL, N_HEADS, N_LAYERS, D_FF = 64, 4, 6, 128
DROPOUT, HEAD_DROPOUT   = 0.2, 0.2
USE_SCALE_FEATURES      = True      # feed per-window mean/log-std back into the head
NORM                    = "batch"   # the paper's choice for time series

# ---- training ----
EPOCHS, BATCH_SIZE = 30, 512
LR, WEIGHT_DECAY   = 3e-4, 1e-4
PATIENCE           = 5              # epochs without a val improvement before stopping
SEED               = 42

# ---- inference benchmark ----
BENCH_BATCH   = 1024        # batch used for the throughput half of the benchmark
BENCH_THREADS = 1           # CPU threads per model; see benchmark_inference.py


## 4. Build the 1-second bar series

One streaming pass folds ~72M ticks into ~2.7M bars. Bars land on a **complete**
1-second grid — trade-less seconds carry a forward-filled price and zero volume — so
a 60-bar horizon is always literally 60 seconds. (On BTC/USDT roughly a fifth of all
seconds contain no trade at all, which is why this matters.)

The result is cached, so re-running is instant and later sweeps never re-read the CSV.

In [ ]:
import time

suffix     = "full" if HOURS is None else f"{HOURS:g}h"
BARS_CACHE = f"{DRIVE_DIR if USE_DRIVE else DATA_DIR}/bars_{SYMBOL}_{MONTH}_{suffix}.npz"

started = time.perf_counter()
if os.path.exists(BARS_CACHE):
    bars = seq.load_bars(BARS_CACHE)
    print(f"loaded cached bars from {BARS_CACHE}")
else:
    bars = seq.load_second_bars(CSV_PATH, hours=HOURS, skip_hours=SKIP_HOURS)
    seq.save_bars(bars, BARS_CACHE)
    print(f"built and cached in {time.perf_counter() - started:.1f}s")

meta = bars["meta"]
print(f"{meta['n_bars']:,} bars | {meta['empty_seconds']:,} trade-less seconds "
      f"({meta['empty_seconds'] / meta['n_bars']:.1%}) filled")

In [ ]:
dataset = seq.build_sequence_dataset(
    bars,
    window=WINDOW,
    horizon=HORIZON,
    barrier=BARRIER,
    label_mode=LABEL_MODE,
    channel_set=CHANNEL_SET,
    train_frac=TRAIN_FRAC,
    val_frac=VAL_FRAC,
    train_stride=TRAIN_STRIDE,
    val_stride=VAL_STRIDE,
)
print(dataset.summary())


## 5. Batchers and model

Windows are never materialised. The `(M, n_bars)` channel matrix goes onto the GPU
once — ~21 MB for a full month at two channels — and each batch is cut from it with
a single gather. Materialising the windows instead would cost `n_windows x M x 300`
float32s: several GB, for data that is 99.7% duplicated between neighbours.

The default `CHANNEL_SET = "raw"` feeds two channels, `log_return` and `ofi`.
Everything the tabular pipeline adds on top (r^2, (pi/2)|r||r_prev|, log volume)
is a pointwise function of those two, so the encoder can form it in its first
layer; handing it over explicitly would spend width to buy nothing. The saved
budget goes into depth instead — 6 encoder layers rather than 3.


In [ ]:
device = pick_device()

train_batcher = WindowBatcher(
    dataset.channels, dataset.splits["train"], dataset.labels("train"), dataset.window, device
)
# `like` reuses the same device-resident channel matrix — no second copy.
val_batcher  = train_batcher.like(dataset.splits["val"],  dataset.labels("val"))
test_batcher = train_batcher.like(dataset.splits["test"], dataset.labels("test"))

config = PatchTSTConfig(
    n_channels=dataset.channels.shape[1],
    seq_len=dataset.window,
    patch_len=PATCH_LEN,
    stride=PATCH_STRIDE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT,
    head_dropout=HEAD_DROPOUT,
    use_scale_features=USE_SCALE_FEATURES,
    norm=NORM,
)
model = PatchTSTClassifier(config).to(device)

print(f"device        : {device}")
print(f"channels      : {config.n_channels} ({', '.join(dataset.meta['channels'])})")
print(f"tokens/channel: {config.num_patches} patches of {config.patch_len} bars, stride {config.stride}")
print(f"parameters    : {model.n_parameters():,}")
print(f"channel matrix: {train_batcher.channels.numel() * 4 / 1e6:.1f} MB resident on {device}")

### Throughput probe

Times real training steps so you know what an epoch costs *before* committing a
session to it. If the estimate is uncomfortable, raise `TRAIN_STRIDE` or lower
`EPOCHS` and re-run from the config cell.

`estimate_throughput` drives the *same* `_training_step` that `fit` uses — same
autocast, same gradient clipping, same per-step host sync — on a throwaway copy of
the model. An earlier hand-written version of this probe left those out and was
optimistic by about 2x.

The estimate still covers training only; add roughly 15–25% for the validation pass
each epoch.

In [ ]:
train_config = TrainConfig(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    early_stop_metric="roc_auc",
    seed=SEED,
    amp=True,
)

rate    = estimate_throughput(model, train_batcher, train_config)
epoch_s = len(train_batcher) / rate
print(f"{rate:,.0f} windows/s  ->  ~{epoch_s:.0f}s per training epoch ({epoch_s / 60:.1f} min)")
print(f"upper bound for {EPOCHS} epochs: ~{epoch_s * EPOCHS / 60:.0f} min "
      f"(early stopping usually ends it sooner)")

## 6. Train

`pos_weight` is derived from the training split exactly the way `train_xgboost.py`
derives `scale_pos_weight`. Early stopping tracks validation ROC-AUC — with a ~7%
positive class, validation *accuracy* would happily reward a model that never fires.

The best-scoring weights are restored before this returns, so what gets exported is
the best epoch, not the last one.

In [ ]:
train_meta = fit(model, train_batcher, val_batcher, train_config, device=device)

## 7. Training curves

Two panels rather than two y-axes — loss and ROC-AUC do not share a scale. The
shape to watch for is the one the PatchTST paper reports for channel-*mixing*
models (Figure 7): training loss still falling while the validation curve has
already turned. The training loss here is the positive-weighted objective, so read
it for its trend, not its level.

In [ ]:
import matplotlib.pyplot as plt

SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e2"
TRAIN_HUE, VAL_HUE = "#2a78d6", "#eb6834"

history = train_meta["history"]
epochs  = [h["epoch"] for h in history]
has_val = "val_roc_auc" in history[0]

fig, (ax_loss, ax_auc) = plt.subplots(1, 2, figsize=(11, 3.8), facecolor=SURFACE)
for ax in (ax_loss, ax_auc):
    ax.set_facecolor(SURFACE)
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#d5d4d0")
    ax.spines["bottom"].set_color("#d5d4d0")
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.set_xlabel("epoch", color=MUTED, fontsize=9)

ax_loss.plot(epochs, [h["train_loss"] for h in history], color=TRAIN_HUE, linewidth=2)
ax_loss.set_title("Training loss  (positive-weighted BCE)", color=INK, fontsize=11, loc="left")

if has_val:
    auc  = [h["val_roc_auc"] for h in history]
    best = train_meta["best_epoch"]
    ax_auc.margins(y=0.22)          # headroom so the peak label clears the title
    ax_auc.axhline(0.5, color="#b9b8b4", linewidth=1, linestyle=(0, (4, 3)))
    ax_auc.annotate("coin flip", xy=(epochs[0], 0.5), xytext=(0, 5),
                    textcoords="offset points", color=MUTED, fontsize=8)
    ax_auc.plot(epochs, auc, color=VAL_HUE, linewidth=2)
    ax_auc.plot([best], [auc[best - 1]], marker="o", markersize=7, color=VAL_HUE,
                markeredgecolor=SURFACE, markeredgewidth=2)
    ax_auc.annotate(f"best {auc[best - 1]:.4f}  (epoch {best})", xy=(best, auc[best - 1]),
                    xytext=(0, 12), textcoords="offset points", ha="center", color=INK, fontsize=9)
ax_auc.set_title("Validation ROC-AUC", color=INK, fontsize=11, loc="left")

fig.tight_layout()
plt.show()

## 8. Score the held-out split

The test split is the final 20% of the period, never touched during training or
early stopping, and separated from the validation split by a purge of
`WINDOW + HORIZON - 1` windows so no training label is drawn from a price move
inside a test lookback.

Under the triple barrier the classes are near-balanced, so 0.5 is a sensible
threshold and accuracy is readable for once. ROC-AUC is still the number to lead
with: it does not depend on where the cut is put.


In [ ]:
probabilities = predict_proba(model, test_batcher, batch_size=4096)
truth         = test_batcher.numpy_labels()
test_metrics  = binary_metrics(truth, probabilities)

print("=========================================")
print(format_metrics(test_metrics))
print("=========================================")
print(f"windows: {test_metrics['n_windows']:,} | positives: {test_metrics['n_positive']:,} "
      f"| flagged: {test_metrics['n_flagged']:,}")

print(f"\n{'p>=':>6} {'flagged':>9} {'precision':>10} {'recall':>8} {'F1':>7}")
for row in threshold_sweep(truth, probabilities):
    print(f"{row['threshold']:>6.2f} {row['n_flagged']:>9,} {row['precision']:>10.4f} "
          f"{row['recall']:>8.4f} {row['f1']:>7.4f}")
print(f"{'(base rate)':>17} {test_metrics['base_rate']:>10.4f}")

## 9. Inference cost — and a like-for-like accuracy comparison

Fits the Logistic Regression and XGBoost baselines on **exactly these windows** —
same 1-second grid, same splits, same labels — then times all three models.

Two numbers, because they rank the models differently:

* **batch = 1 latency** is the trading number: what a signal costs when a window
  closes and you must decide before the next tick.
* **batched throughput** is the research number: what a backtest over a month costs.

Every CPU model is pinned to one thread, so the comparison measures models rather
than core counts. The tabular models additionally pay a feature-preparation step
(300 bars → 7 scalars) that PatchTST does not — it reads the window directly and
normalises inside the forward pass — so that is reported separately and should be
added to their latencies.

Because the baselines have to be fitted to be timed, their test ROC-AUC comes out
for free — and unlike the README's historical table, it is measured on windows
identical to PatchTST's.

In [ ]:
from benchmark_inference import run_benchmark, format_benchmark

bench = run_benchmark(
    dataset,
    model,
    bars,
    device=device,
    batch_size=BENCH_BATCH,
    batch_repeats=10,
    threads=BENCH_THREADS,
)
print(format_benchmark(bench))

print("\nTest ROC-AUC on identical windows")
print("-" * 42)
for name, score in bench["accuracy"].items():
    print(f"  {name:<24} {score:.4f}")
print(f"  {'PatchTST':<24} {test_metrics['roc_auc']:.4f}")
print(f"  {'(base rate)':<24} {test_metrics['base_rate']:.4f}")

## 10. Export the weights

The checkpoint carries the weights **plus the recipe**: model config, the data slice
it was trained on, split fractions, and the test metrics. `eval_patchtst.py` reads
that recipe to rebuild the identical window population locally — so if the local run
disagrees with the numbers above, it tells you instead of silently scoring something
else.

It is a few hundred KB. `files.download` opens a browser download; the Drive copy is
the backup for when Colab times out mid-session.

In [ ]:
import shutil

WEIGHT_NAME = f"patchtst_{SYMBOL}_{MONTH}_{suffix}.pt"
checkpoint  = save_checkpoint(
    f"/content/{WEIGHT_NAME}",
    model,
    data_meta=dataset.meta,
    train_meta=train_meta,
    metrics={**test_metrics, "benchmark_auc": bench["accuracy"]},
)
print(f"{checkpoint}  ({checkpoint.stat().st_size / 1e6:.2f} MB)")

if USE_DRIVE:
    shutil.copy(checkpoint, f"{DRIVE_DIR}/{WEIGHT_NAME}")
    print(f"copied to {DRIVE_DIR}/{WEIGHT_NAME}")

from google.colab import files
files.download(str(checkpoint))

## 11. Run it locally

Move the downloaded file into the repo's `weights/` directory, then:

```bash
uv sync                       # first time only: pulls torch
cd python
python eval_patchtst.py --weights ../weights/<the-file>.pt
```

It rebuilds the same bars from your local CSV, scores the same held-out split, prints
the metrics table, and appends a row to `python/training_logs.txt` alongside the
XGBoost and logistic runs. Rebuilding the month's bars takes ~30s the first time;
pass `--bars-cache ../data/bars_full.npz` to reuse it afterwards.

Useful flags: `--device mps` (Apple Silicon), `--split val`, `--threshold 0.7`,
`--save-predictions probs.npy`.

The inference benchmark also runs locally:

```bash
python benchmark_inference.py --weights ../weights/<the-file>.pt \
                              --bars-cache ../data/bars_full.npz
```

In [ ]:
print("Local command:")
print(f"  cd python && python eval_patchtst.py --weights ../weights/{WEIGHT_NAME}")